# Lunar Lander RL — Colab training notebook

Train, evaluate and visualise the **baseline** (`plain`), **reward-hack** (`hack`) and **potential-based fix** (`fix`) PPO variants on Gymnasium `LunarLander-v3`.

This notebook calls the repo's own scripts rather than reimplementing them:

- `src/train.py` — PPO `MlpPolicy`, optional reward wrapper, `--run-name` default `<wrapper>_s<seed>`
- `src/evaluate.py` — deterministic evaluation on the **plain** (unwrapped) env
- `src/record.py` — GIF (and optional MP4) rollouts on the plain env
- `src/analysis/plot_curves.py` — combined learning curves from Monitor CSVs (plain-env reward)

Models are written to `models/`, logs to `logs/`, eval JSON to `results/`, and clips / curve plots to `assets/`. On Colab, a later cell copies `models/` and `results/` to Google Drive.

**GPU note:** a GPU rarely helps small-MLP PPO (the 2×64 policy is tiny; the bottleneck is CPU env stepping). Leave `DEVICE='cpu'` even on a CUDA runtime unless you are experimenting.

To run locally, open this notebook from a checkout of the repo in a venv with `requirements.txt` installed. Colab-only cells (clone, `pip install`, Drive save) no-op when `google.colab` is not in `sys.modules`. Override `TIMESTEPS`, `SEEDS`, `DEVICE`, `N_ENVS` and the eval/record knobs with environment variables for a short smoke test.


In [ ]:
# Configuration. Environment variables override the defaults so a local
# smoke test can drop TIMESTEPS / SEEDS without editing the notebook.
import os
import sys
from pathlib import Path

# Placeholder until the repo is pushed. Replace <your-username> with yours.
REPO_URL = os.environ.get(
    "REPO_URL",
    "https://github.com/<your-username>/lunar-lander-rl.git",
)

TIMESTEPS = int(os.environ.get("TIMESTEPS", "1000000"))
SEEDS = [int(s) for s in os.environ.get("SEEDS", "0,1").split(",") if s.strip()]
DEVICE = os.environ.get("DEVICE", "cpu")
# Default matches src/train.py: min(8, nproc). Override for a short smoke test.
N_ENVS = int(os.environ.get("N_ENVS", str(min(8, os.cpu_count() or 1))))
EVAL_EPISODES = int(os.environ.get("EVAL_EPISODES", "20"))
RECORD_EPISODES = int(os.environ.get("RECORD_EPISODES", "2"))
SKIP_MP4 = os.environ.get("SKIP_MP4", "0").strip() not in {"0", "false", "False", ""}
PLOT_WINDOW = int(os.environ.get("PLOT_WINDOW", "100"))
VARIANTS = ["plain", "hack", "fix"]  # baseline, reward-hack, potential-based fix

# Auto-detect Colab. All Colab-only cells check this flag and no-op locally.
RUN_IN_COLAB = "google.colab" in sys.modules

# The kernel cwd is often the notebooks/ folder; move to the repo root.
cwd = Path.cwd().resolve()
if (cwd / "src" / "train.py").is_file():
    REPO_ROOT = cwd
elif (cwd.parent / "src" / "train.py").is_file():
    REPO_ROOT = cwd.parent
    os.chdir(REPO_ROOT)
else:
    REPO_ROOT = cwd

print(f"RUN_IN_COLAB={RUN_IN_COLAB}")
print(f"REPO_ROOT={REPO_ROOT}")
print(f"REPO_URL={REPO_URL}")
print(f"TIMESTEPS={TIMESTEPS}  SEEDS={SEEDS}  DEVICE={DEVICE}  N_ENVS={N_ENVS}")
print(f"EVAL_EPISODES={EVAL_EPISODES}  RECORD_EPISODES={RECORD_EPISODES}  SKIP_MP4={SKIP_MP4}")
print(
    "Note: a GPU rarely helps small-MLP PPO; DEVICE='cpu' is the recommended default."
)


## Clone the repo (Colab only)

Skipped when `src/train.py` is already on the path (local checkout, or a previous cell already cloned it). On Colab, replace the `REPO_URL` placeholder in the config cell first, or set the `REPO_URL` environment variable.


In [ ]:
import os
import subprocess
from pathlib import Path

if not RUN_IN_COLAB:
    print("Skipping clone (not running in Colab; using the local checkout).")
else:
    if (Path.cwd() / "src" / "train.py").is_file():
        print(f"Already in the repo at {Path.cwd()}, skipping clone.")
    else:
        dest = Path("/content/lunar-lander-rl")
        if not (dest / "src" / "train.py").is_file():
            if "<your-username>" in REPO_URL:
                raise ValueError(
                    "REPO_URL is still the placeholder. Edit the config cell "
                    "or set the REPO_URL environment variable to your fork."
                )
            subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(dest)])
        os.chdir(dest)
        print(f"Working directory is now {Path.cwd()}")


## Install dependencies (Colab only)

`swig` is installed first because `gymnasium[box2d]` / Box2D may need it to compile the Box2D bindings when a prebuilt wheel is not available. Then `requirements.txt` is installed (pinned versions, including `torch==2.15.0` without a `+cpu` suffix).

On Linux, a plain PyPI `torch` wheel already includes CUDA libraries, which is fine on Colab or an NVIDIA laptop (e.g. RTX 4050) but is a large download. For a CPU-only machine see the header comments in `requirements.txt`.


In [ ]:
import subprocess
import sys
from pathlib import Path

if not RUN_IN_COLAB:
    print("Skipping pip install (not running in Colab; using the existing venv).")
else:
    req = Path("requirements.txt")
    if not req.is_file():
        raise FileNotFoundError(
            "requirements.txt not found. Run the clone cell first, or start "
            "the notebook from the repo root."
        )
    # swig first so a from-source Box2D build can find the binary.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "swig==4.5.1"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)])
    print("Installed swig and requirements.txt")


## Train baseline, hack and fix

Each `(wrapper, seed)` pair calls `src/train.py` with the Trainer's CLI. `hack` and `fix` use the defaults from `src/wrappers.py` (no extra `--wrapper-kwargs`). Run names default to `<wrapper>_s<seed>`, e.g. `plain_s0`, `hack_s1`, `fix_s0`.


In [ ]:
import subprocess
import sys

python = sys.executable
train_runs = []
for wrapper in VARIANTS:
    for seed in SEEDS:
        run_name = f"{wrapper}_s{seed}"
        cmd = [
            python,
            "src/train.py",
            "--wrapper",
            wrapper,
            "--seed",
            str(seed),
            "--timesteps",
            str(TIMESTEPS),
            "--device",
            DEVICE,
            "--n-envs",
            str(N_ENVS),
            "--run-name",
            run_name,
            "--torch-threads",
            "1",
            "--verbose",
            "0",
        ]
        print(" ".join(cmd), flush=True)
        subprocess.check_call(cmd)
        train_runs.append(run_name)

print("Finished training:", train_runs)


## Evaluate on the plain environment

`src/evaluate.py` never applies a reward wrapper. It reports mean / std return, landing rate (terminated with the +100 at-rest bonus), solved rate (return ≥ 200), crash rate and timeout rate over `EVAL_EPISODES` deterministic episodes (default 20). Results go to `results/<run>_eval.json`.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

from IPython.display import Markdown, display

python = sys.executable
eval_rows = []
for run_name in train_runs:
    model = Path("models") / f"{run_name}.zip"
    cmd = [
        python,
        "src/evaluate.py",
        "--model",
        str(model),
        "--episodes",
        str(EVAL_EPISODES),
        "--device",
        DEVICE,
    ]
    print(" ".join(cmd), flush=True)
    subprocess.check_call(cmd)
    result_path = Path("results") / f"{run_name}_eval.json"
    result = json.loads(result_path.read_text())
    eval_rows.append(result)
    print(
        f"  {run_name}: mean={result['mean_return']:.2f} ± {result['std_return']:.2f} "
        f"land+100={result['landing_rate']:.3f} solved={result['solved_rate']:.3f} "
        f"crash={result['crash_rate']:.3f} timeout={result['truncated_rate']:.3f} "
        f"len={result['mean_length']:.1f}",
        flush=True,
    )

lines = [
    "| Run | Mean return | Std | Land (+100) | Solved (≥200) | Crash | Timeout | Mean length |",
    "| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |",
]
for row in eval_rows:
    name = Path(row["model"]).stem
    lines.append(
        f"| {name} | {row['mean_return']:.2f} | {row['std_return']:.2f} | "
        f"{row['landing_rate']:.3f} | {row['solved_rate']:.3f} | "
        f"{row['crash_rate']:.3f} | {row['truncated_rate']:.3f} | "
        f"{row['mean_length']:.1f} |"
    )
display(Markdown("\n".join(lines)))


## Learning curves

Monitor CSVs sit *under* the reward wrappers, so every curve is plain-env reward and the variants are comparable. The plot is also shown inline.


In [ ]:
import subprocess
import sys
from pathlib import Path

from IPython.display import Image, display

python = sys.executable
curve_png = Path("assets") / "colab_learning_curves.png"
cmd = [
    python,
    "src/analysis/plot_curves.py",
    "--output",
    str(curve_png),
    "--window",
    str(PLOT_WINDOW),
    "--title",
    "Lunar Lander learning curves (plain-env reward)",
]
for wrapper, label in (("plain", "baseline"), ("hack", "hack"), ("fix", "fix")):
    cmd.extend(["--label", f"{label}=logs/{wrapper}_s*/*.monitor.csv"])
print(" ".join(cmd), flush=True)
subprocess.check_call(cmd)
display(Image(filename=str(curve_png)))


## Rollout GIFs

`src/record.py` writes `assets/<run>.gif` (kept under 5 MB) and, unless `SKIP_MP4` is set, a full-resolution MP4. GIFs are displayed inline.


In [ ]:
import subprocess
import sys
from pathlib import Path

from IPython.display import Image, display

python = sys.executable
for run_name in train_runs:
    cmd = [
        python,
        "src/record.py",
        "--model",
        str(Path("models") / f"{run_name}.zip"),
        "--episodes",
        str(RECORD_EPISODES),
        "--device",
        DEVICE,
        "--name",
        run_name,
    ]
    if SKIP_MP4:
        cmd.append("--no-mp4")
    print(" ".join(cmd), flush=True)
    subprocess.check_call(cmd)
    gif = Path("assets") / f"{run_name}.gif"
    print(f"  {gif} ({gif.stat().st_size / 1e6:.2f} MB)")
    display(Image(filename=str(gif)))


## Save models and results to Google Drive (Colab only)

Mounts Drive and copies `models/` and `results/` to `Drive/lunar-lander-rl/`. No-ops when not running in Colab.


In [ ]:
import shutil
from pathlib import Path

if not RUN_IN_COLAB:
    print("Skipping Google Drive save (not running in Colab).")
else:
    from google.colab import drive

    drive.mount("/content/drive")
    dest = Path("/content/drive/MyDrive/lunar-lander-rl")
    dest.mkdir(parents=True, exist_ok=True)
    for folder in ("models", "results"):
        src = Path(folder)
        if not src.is_dir():
            print(f"No {folder}/ directory to copy.")
            continue
        target = dest / folder
        shutil.copytree(src, target, dirs_exist_ok=True)
        print(f"Copied {src} -> {target}")
